# CS50 Section 5

## 5.1 Choosing Data Structures and Trade-offs

### 5.1.1 Section Overview and the Wake-Word Scenario

Week 5 shifts focus from single variables and contiguous arrays to custom **data structures** built from pointers and dynamic memory.

Three core engineering questions govern the material in this section:

- What **trade-offs** govern the choice of one data structure over another?
- How are the primary operations — creation, head insertion, traversal and memory deallocation — implemented on a **linked list**?
- How is a **hash table** structured?
- How is a **hash function** written and refined in C?

In high-level languages such as R, data containers like `c()`, `list()` and `data.frame()` manage memory growth and lookup mechanics automatically behind the scenes.

In C, no such automatic container exists. Every structural layout and resizing behaviour must be chosen and wired explicitly by the programmer.

Consider a concrete engineering scenario involving a digital voice assistant running on a mobile device.

Customer reports show that restricting activation to a single **wake word** (such as `"Hello"`) causes frequent missed activations.

Supporting a broader set of wake words and phrases — such as `"Hello"`, `"Hi"`, `"Hey"`, `"Lo"`, `"Lo there"` and `"Hello there"` — requires selecting an underlying data structure to store and query those strings efficiently.

### 5.1.2 Prioritising Operations: Search, Insertion and Deletion

Selecting an appropriate data structure begins with identifying the primary operations the application performs on its stored collection.

Most collection workloads revolve around three fundamental operations:

- **Deletion**: removing obsolete wake words from the stored vocabulary.
- **Insertion**: adding newly supported wake words into the structure.
- **Search**: checking whether a spoken phrase matches any stored wake word.

No single data structure optimises all three operations simultaneously without cost. Engineering teams therefore rank those operations by frequency and latency sensitivity:

| Priority Rank | Search-Heavy Workload (e.g. Live Wake-Word Detection) | Insertion-Heavy Workload (e.g. Live Data Logging) |
|---|---|---|
| 1 | **Search** — executed continuously on every spoken phrase | **Insertion** — executed constantly as new entries stream in |
| 2 | **Insertion** — executed occasionally during vocabulary updates | **Search** — executed only during periodic lookups |
| 3 | **Deletion** — executed rarely when retiring phrases | **Deletion** — executed rarely during cleanup |

When live wake-word recognition runs on every utterance, **search** speed dominates the user experience.

Conversely, when an application spends most of its time recording new items and rarely queries them, **insertion** speed takes priority.

### 5.1.3 Candidate Structures: Linked Lists, Hash Tables and Tries

Three pointer-based data structures illustrate distinct ways to store the wake phrases `"Hello"`, `"Hey"` and `"Lo there"`.

#### 5.1.3.1 Linked List

A **linked list** stores elements in individual heap-allocated **nodes** chained together via memory addresses.

A single head pointer (`list`) stores the address of the first node. The final node's pointer stores `NULL` to mark the end of the chain:

```text
list
  |
  v
+-----------+     +-----------+     +------------+
|  "Hello"  |     |   "Hey"   |     | "Lo there" |
+-----------+     +-----------+     +------------+
|     *-----+---->|     *-----+---->|    NULL    |
+-----------+     +-----------+     +------------+
```

A linked list behaves like a **scavenger hunt** across lockers in a hallway.

A sticky note on the desk (`list`) gives only the first locker number. Opening that locker reveals `"Hello"` plus a card naming the next locker's number, continuing until the last locker displays `NULL`.

#### 5.1.3.2 Hash Table

A **hash table** combines the instant random access of an array with the dynamic growth of linked lists.

Each slot of an array acts as a labelled **bucket** (for example, one bucket per starting letter `A`–`Z`). Each bucket holds the head pointer to its own shorter linked list:

```text
Bucket
+-----+
| ... |
+-----+     +---------+     +-------+
|  H  |---->| "Hello" |---->| "Hey" |----> NULL
+-----+     +---------+     +-------+
|  I  |----> NULL
+-----+
|  J  |----> NULL
+-----+
|  K  |----> NULL
+-----+     +------------+
|  L  |---->| "Lo there" |----> NULL
+-----+     +------------+
| ... |
+-----+
```

In R terms, a hash table resembles an R `environment` created via `new.env(hash = TRUE)`. Looking up a key jumps straight to its designated slot instead of scanning every stored element from start to finish.

Physically, the array acts like a wall of labelled mailroom pigeonholes. `"Hello"` and `"Hey"` land in slot `H`, whereas `"Lo there"` lands in slot `L`.

#### 5.1.3.3 Trie

A **trie** (retrieval tree) organises strings character by character along branching tree paths starting from an empty root node.

Words that share a common prefix (such as `"HELLO"` and `"HEY"`, which share `'H'` followed by `'E'`) share the same upper nodes before branching apart:

```text
                  (L) ---> (L) ---> (O)                 ["HELLO"]
                 /
        (H) -> (E)
       /         \
(root)            (Y)                                   ["HEY"]
       \
        (L) -> (O) -> (␣) -> (T) -> (H) -> (E) -> (R) -> (E)   ["LO THERE"]
```

Searching a trie steps down one link per character in the target word. Lookup time therefore scales with the length of the word rather than the total number of stored words.

### 5.1.4 The Time-Space Trade-off: The Speller Big Board

Choosing among a linked list, a hash table and a trie exposes the central engineering trade-off in data-structure design: **time** (execution speed) versus **space** (memory consumption).

Problem Set 5 (`speller`) benchmarks dictionary implementations on a leaderboard ("Big Board") across two primary metrics:

- **Time**: the sum of `Load`, `Check`, `Size` and `Unload` durations (in seconds) required to spell-check `texts/*.txt` against `dictionaries/large`.
- **Memory**: the peak combined `Heap` and `Stack` utilisation required when spell-checking `texts/holmes.txt` against `dictionaries/large`.

Representative top entries from a past `speller` Big Board illustrate these metrics side by side:

| Rank | Total Time | Load | Check | Size | Unload | Peak Memory | Heap |
|---|---|---|---|---|---|---|---|
| 1 (Staff) | `6.136 s` | `1.234 s` | `4.902 s` | `0.000 s` | `0.000 s` | `12.3 kB` | `4.6 kB` |
| 2 | `7.119 s` | `0.932 s` | `5.651 s` | `0.000 s` | `0.536 s` | `8.0 MB` | `8.0 MB` |
| 3 | `10.248 s` | `1.079 s` | `8.319 s` | `0.000 s` | `0.850 s` | `8.0 MB` | `8.0 MB` |

> **General Time-Space Rule:** Optimising a data structure for faster lookups almost always consumes additional memory — for instance, allocating thousands of hash buckets or wide trie node arrays reaching `8.0 MB` on the heap (roughly 650 times larger than `12.3 kB`). Conversely, minimising memory footprint by packing items into fewer buckets or a single list typically lengthens `Check` time.

### 5.1.5 Data Structure Selection Cheat Sheet

| Structure | Underlying Layout | Primary Strength | Primary Cost |
|---|---|---|---|
| **Linked List** | Chain of heap `node` structs linked by `next` pointers | Compact memory usage; fast `O(1)` insertion at the head | Slow `O(n)` linear search across all nodes |
| **Hash Table** | Fixed array of buckets, each heading a linked list | Much faster search by jumping directly to one bucket | Extra memory for the bucket array (including empty slots); collisions still require short list scans |
| **Trie** | Tree of character-indexed pointer arrays | Very fast prefix/word lookup scaled only by word length | Very large memory footprint due to wide pointer arrays at every node |

### 5.1.6 Questions

1. A mobile voice assistant checks thousands of spoken audio transcriptions per day against a fixed set of 500 wake phrases that changes only once per software release. How should **search**, **insertion** and **deletion** be prioritised? Why does that ranking argue against a plain linked list?
2. In the trie storing `"HELLO"`, `"HEY"` and `"LO THERE"`, how many nodes below the root store the prefix `"HE"`? Why are those nodes not duplicated for each word?
3. What is the fundamental resource trade-off when increasing the number of buckets in a hash table or switching from a linked list to a trie?

## 5.2 Linked Lists: Nodes, Creation and Head Insertion

### 5.2.1 Anatomy of a Node Struct

A linked list is built from custom structures called **nodes**. Each node bundles two pieces of information: the stored value and a pointer to the next node in line.

In R, a table of linked records can be modelled as a two-column `data.frame` where column 1 holds a text string and column 2 holds the integer row number of the next item.

In C, that two-compartment container is defined with `typedef struct node`:

```c
typedef struct node
{
    string phrase;
    struct node *next;
}
node;
```

Each line of this type definition plays a distinct syntactic role:

1. `typedef struct node` begins defining a structure tagged `struct node` so the type can refer to itself inside its own body.
2. `string phrase;` declares the first compartment (the payload), capable of holding a string address such as `"Hi!"`.
3. `struct node *next;` declares the second compartment, a pointer holding the heap memory address (such as `0x123`) of the next `node` in the chain. Writing `struct node` in full is mandatory here because the shorthand alias `node` does not exist until line 6.
4. `} node;` closes the definition and establishes `node` as the short type alias for `struct node` everywhere below.

```text
       node
+----------------+
|     "Hi!"      |  <-- string phrase   (payload)
+----------------+
|     0x123      |  <-- struct node *next (address of next node)
+----------------+
```

### 5.2.2 Creating a Single-Node Linked List

Constructing a linked list from scratch proceeds one pointer step at a time.

Tracing the five-line creation sequence for a single node storing `"Hi!"`:

```c
node *list = NULL;

node *n = malloc(sizeof(node));
n->phrase = "Hi!";
n->next = NULL;

list = n;
```

**Step 1 — Initialise an empty list head (`node *list = NULL;`):**

A pointer variable `list` is declared on the stack and initialised to `NULL`.

Because no nodes exist yet, `NULL` explicitly marks the list as empty rather than leaving `list` holding a random garbage address:

```text
list
  |
  v
+------+
| NULL |
+------+
```

**Step 2 — Dynamically allocate one node (`node *n = malloc(sizeof(node));`):**

`malloc(sizeof(node))` asks the operating system for enough contiguous heap bytes to hold one `node` struct (both its `phrase` field and its `next` pointer).

`malloc` returns the starting memory address of that newly leased heap block (say `0x100`), which is stored in temporary pointer `n`:

```text
list                      n
  |                       |
  v                       v
+------+                +----------+  (0x100)
| NULL |                |    ?     |  phrase
+------+                +----------+
                        |    ?     |  next
                        +----------+
```

**Step 3 — Populate the payload (`n->phrase = "Hi!";`):**

The arrow operator (`->`) dereferences pointer `n` and accesses the `phrase` field inside the heap block (`(*n).phrase`), storing `"Hi!"`:

```text
list                      n
  |                       |
  v                       v
+------+                +----------+  (0x100)
| NULL |                |  "Hi!"   |  phrase
+------+                +----------+
                        |    ?     |  next
                        +----------+
```

**Step 4 — Terminate the node's `next` pointer (`n->next = NULL;`):**

Because this node is currently the only element in the list, nothing follows it.

Setting `n->next = NULL` overwrites the uninitialised garbage bytes in `next` with `NULL`, marking a clean end-of-list boundary:

```text
list                      n
  |                       |
  v                       v
+------+                +----------+  (0x100)
| NULL |                |  "Hi!"   |  phrase
+------+                +----------+
                        |   NULL   |  next
                        +----------+
```

**Step 5 — Point the head at the new node (`list = n;`):**

Copying the address stored in `n` (`0x100`) into `list` makes the new node the official **head** of the linked list:

```text
                list, n
                   |
                   v
                +----------+  (0x100)
                |  "Hi!"   |  phrase
                +----------+
                |   NULL   |  next
                +----------+
```

### 5.2.3 Setting Up list.c: Headers, Allocation and Defensive Checks

To put linked-list creation into practice, the program `list.c` builds a linked list of three integers entered by the user.

Three standard header files supply the required declarations:

- `#include <cs50.h>` provides `get_int` for reading user input.
- `#include <stdio.h>` provides `printf` for terminal output.
- `#include <stdlib.h>` provides `malloc`, `free` and `NULL` for dynamic memory management.

Because `list.c` stores integers rather than strings, the `node` struct replaces `string phrase` with `int number`:

```c
typedef struct node
{
    int number;
    struct node *next;
} node;
```

Rather than calling `get_int` three separate times into `number1`, `number2` and `number3`, a `for` loop (`for (int i = 0; i < 3; i++)`) repeats the prompt and node allocation cleanly.

> **Design Note — Magic Numbers:** Writing the literal `3` directly inside `i < 3` is convenient in a short demonstration file. Production programs and problem sets instead define a named constant (such as `#define LIST_SIZE 3` or `const int`) to avoid unexplained **magic numbers**.

#### 5.2.3.1 Checking `malloc` for `NULL` and Mid-Loop Failures

Whenever `malloc(sizeof(node))` is called, the system may fail to find free heap space and return `NULL` instead of a valid address.

In defensive C code, `n` is checked immediately after allocation before `n->number` is ever dereferenced:

```c
node *n = malloc(sizeof(node));
if (n == NULL)
{
    printf("malloc failed\n");
    return 1;
}
```

> **Gotcha — Mid-Loop Allocation Failure:** Consider a run where `malloc` succeeds on iteration `i = 0` (allocating node 1 and linking it to `list`) before returning `NULL` on iteration `i = 1`. Returning `1` immediately exits `main` while node 1 is still sitting allocated on the heap! A complete error handler inside a loop must free any nodes already attached to `list` before returning `1` (the final `list.c` exercise omits the mid-loop `if (n == NULL)` block to keep focus on linking and end-of-program freeing).

### 5.2.4 Prepending Nodes at the Head

Once `list` already points to an existing node (`"Hi!"`), inserting a second node (`"Hey!"`) at the **head** of the list requires a strict pointer-wiring order:

```c
n = malloc(sizeof(node));
n->phrase = "Hey!";
n->next = list;
list = n;
```

Tracing those four lines step by step when `list` currently points to `"Hi!"` at `0x100`:

1. `n = malloc(sizeof(node));` allocates a fresh node at `0x200` and points `n` at `0x200`, while `list` still points safely at `0x100` (`"Hi!"`).
2. `n->phrase = "Hey!";` writes `"Hey!"` into the new node at `0x200`.
3. `n->next = list;` copies the current head address (`0x100`) into the new node's `next` field, tethering `"Hey!"` ahead of `"Hi!"`:

```text
  n                      list
  |                        |
  v                        v
+----------+  (0x200)    +----------+  (0x100)
|  "Hey!"  |             |  "Hi!"   |
+----------+             +----------+
|  0x100 --+------------>|   NULL   |
+----------+             +----------+
```

4. `list = n;` updates `list` to `0x200`, promoting `"Hey!"` to be the new head of the two-node chain:

```text
 list
  |
  v
+----------+  (0x200)    +----------+  (0x100)
|  "Hey!"  |             |  "Hi!"   |
+----------+             +----------+
|  0x100 --+------------>|   NULL   |
+----------+             +----------+
```

> **Critical Ordering Trap:** `n->next = list;` **must** execute before `list = n;`. Executing `list = n;` first overwrites `list` with `0x200`, permanently losing the address `0x100` of `"Hi!"` (orphaning the entire existing list). The next line `n->next = list;` then points node `0x200` right back at itself in an infinite loop!

The statement `n->next = list;` also handles the very first node seamlessly.

On iteration `i = 0`, `list` equals `NULL`, so `n->next = list;` automatically sets the first node's `next` field to `NULL` without needing a special `if` branch.

### 5.2.5 Questions

1. Inside `typedef struct node { int number; struct node *next; } node;`, why must line 3 be written `struct node *next;` rather than `node *next;`?
2. Why is `node *list = NULL;` necessary when declaring the head of a brand-new linked list instead of writing `node *list;`?
3. What happens in memory if `list = n;` executes on the line *before* `n->next = list;` during the second iteration of the node-insertion loop?
4. Why does `n->next = list;` work correctly on the very first loop iteration (`i = 0`), when no nodes exist in the list yet?

## 5.3 Linked Lists: Traversal, Freeing and Full list.c Trace

### 5.3.1 Why Freeing Only the Head Orphans the List

Every heap block leased via `malloc` must be returned to the operating system via `free` before the program finishes.

Consider `list` pointing to the two-node chain `"Hey!"` (`0x200`) → `"Hi!"` (`0x100`) → `NULL`, followed by a naive call to `free(list)`:

```c
free(list); // BUG: frees only the first node and orphans the rest!
```

`free(list)` immediately hands block `0x200` back to the heap allocator, rendering its contents invalid:

```text
 list
  |
  v
+----------+  (0x200)    +----------+  (0x100)
| [FREED]  |             |  "Hi!"   |  <-- ORPHANED (memory leak!)
| [FREED]  |             +----------+
+----------+             |   NULL   |
                         +----------+
```

Block `0x200` held the *only* copy of address `0x100` (`next`).

Once `0x200` is freed, `0x100` can no longer be reached to read or `free` it, leaving an **orphaned** node that causes a memory leak.

Attempting to read `list->next` after `free(list)` is equally invalid because it touches already-freed memory (a **use-after-free** error).

In the scavenger-hunt analogy, `free(list)` shreds Clue 1 before the locker number printed on the bottom of Clue 1 is copied down. Clue 2 remains locked away forever with no map leading to it.

### 5.3.2 Safe Freeing with a Temporary Pointer

Freeing a linked list safely requires saving a foothold pointer before calling `free` on each node.

Two equivalent patterns accomplish this cleanly.

#### 5.3.2.1 Pattern A — Temporary Pointer Saves `list->next` (Slide Walkthrough)

In the slide sequence, a helper pointer `ptr` steps ahead to `list->next` *before* `list` is freed. The head pointer `list` then catches up to `ptr`:

```c
// Pass 1: free "Hey!" (0x200) while preserving "Hi!" (0x100)
node *ptr = list->next; // ptr saves 0x100 ("Hi!")
free(list);             // safely frees 0x200 ("Hey!")
list = ptr;             // list moves to 0x100 ("Hi!")

// Pass 2: free "Hi!" (0x100) and reach NULL
ptr = list->next;       // ptr saves NULL
free(list);             // safely frees 0x100 ("Hi!")
list = ptr;             // list becomes NULL (all nodes freed!)
```

#### 5.3.2.2 Pattern B — Temporary Pointer Saves Current Node While `ptr` Advances (`list.c` Walkthrough)

In `list.c`, the cursor pointer `ptr` walks down the list while a block-scoped pointer `tmp` holds the node about to be freed:

```c
node *ptr = list;
while (ptr != NULL)
{
    node *tmp = ptr;
    ptr = ptr->next;
    free(tmp);
}
```

Inside each `while` pass:

1. `node *tmp = ptr;` bookmarks the current node so its address is not lost.
2. `ptr = ptr->next;` advances `ptr` safely to the next node while the current node is still intact in memory.
3. `free(tmp);` frees the bookmarked current node. On the next check of `while (ptr != NULL)`, `ptr` is already waiting at the next node (or `NULL` when the list ends).

> **Common Bug — `free(ptr)` Instead of `free(tmp)`:** Calling `free(ptr);` on the third line (after `ptr = ptr->next;` has already moved `ptr` forward!) frees the *next* node instead of the current one. That mistake leaks the current node, corrupts the next node before its `next` pointer is read and breaks the rest of the loop.

### 5.3.3 Traversing and Printing Nodes in list.c

In R or Python, iterating over a vector uses integer positions (`for (i in seq_along(x))`). Integer indexing works only because array elements sit in adjacent memory slots.

Because linked-list nodes can sit at arbitrary, non-adjacent heap addresses, `list.c` traverses the list using a **node pointer cursor** inside a `for` loop:

```c
for (node *ptr = list; ptr != NULL; ptr = ptr->next)
{
    printf("%i\n", ptr->number);
}
```

Comparing the three clauses of a standard integer `for` loop against a linked-list `for` loop clarifies how pointer traversal works:

| `for` Loop Clause | Array Index Loop | Linked-List Cursor Loop | Role in Traversal |
|---|---|---|---|
| **Initialisation** | `int i = 0` | `node *ptr = list` | Starts the cursor at the head of the list (scoped solely to this `for` loop) |
| **Condition** | `i < 3` | `ptr != NULL` | Continues running as long as `ptr` points to a valid node (`NULL` signals end of list) |
| **Update** | `i++` | `ptr = ptr->next` | Follows the current node's `next` address to hop to the next node in the chain |

Inside the loop body, `ptr->number` reads the integer stored inside whichever node `ptr` currently points to.

### 5.3.4 Complete list.c Implementation and Execution Trace

The complete `list.c` file combines node definition, head-insertion in a loop, pointer traversal and `while`-loop deallocation:

```c
#include <cs50.h>
#include <stdio.h>
#include <stdlib.h>

typedef struct node
{
    int number;
    struct node *next;
} node;

int main(void)
{
    // Head of linked list
    node *list = NULL;

    // Iteratively create nodes
    for (int i = 0; i < 3; i++)
    {
        int x = get_int("Number: ");

        node *n = malloc(sizeof(node));
        n->number = x;
        n->next = list;

        list = n;
    }

    // Traverse through linked list and print each number
    for (node *ptr = list; ptr != NULL; ptr = ptr->next)
    {
        printf("%i\n", ptr->number);
    }

    // Free the linked list
    node *ptr = list;
    while (ptr != NULL)
    {
        node *tmp = ptr;
        ptr = ptr->next;
        free(tmp);
    }
}
```

#### 5.3.4.1 Why Inputs `1`, `2`, `3` Print in Reverse (`3`, `2`, `1`)

Compiling and running `./list` with inputs `1`, `2` and `3` produces:

```text
$ ./list
Number: 1
Number: 2
Number: 3
3
2
1
```

Because every new node is inserted at the **head** (`list = n`), the list behaves in **first-in, last-out** (FILO / LIFO) order — just like stacking plates one on top of another.

Tracing the exact heap state across all three phases (assuming `malloc` returns `0x100`, `0x200` and `0x300`):

| Phase & Step | Executed Lines | `list` | Chain State (`head -> ... -> NULL`) | Terminal Output / Freed Block |
|---|---|---|---|---|
| **Start** | `node *list = NULL;` | `NULL` | `NULL` | — |
| **Insert `i = 0`** (`x = 1`) | `n = 0x100; n->number = 1; n->next = NULL; list = 0x100;` | `0x100` | `[1 \| NULL]` | — |
| **Insert `i = 1`** (`x = 2`) | `n = 0x200; n->number = 2; n->next = 0x100; list = 0x200;` | `0x200` | `[2 \| 0x100] -> [1 \| NULL]` | — |
| **Insert `i = 2`** (`x = 3`) | `n = 0x300; n->number = 3; n->next = 0x200; list = 0x300;` | `0x300` | `[3 \| 0x200] -> [2 \| 0x100] -> [1 \| NULL]` | — |
| **Print Pass 1** | `ptr = 0x300` (`!= NULL`) → `printf(ptr->number)` | `0x300` | Cursor at `0x300` (`3`) | Prints `3` |
| **Print Pass 2** | `ptr = ptr->next` (`0x200`) → `printf(ptr->number)` | `0x300` | Cursor at `0x200` (`2`) | Prints `2` |
| **Print Pass 3** | `ptr = ptr->next` (`0x100`) → `printf(ptr->number)` | `0x300` | Cursor at `0x100` (`1`) | Prints `1` |
| **Print End** | `ptr = ptr->next` (`NULL`) → loop exits | `0x300` | `ptr` scoped to `for` loop expires | — |
| **Free Pass 1** | `tmp = 0x300; ptr = 0x200; free(0x300);` | `0x300` | `[2 \| 0x100] -> [1 \| NULL]` | Frees `0x300` (`3`) |
| **Free Pass 2** | `tmp = 0x200; ptr = 0x100; free(0x200);` | `0x300` | `[1 \| NULL]` | Frees `0x200` (`2`) |
| **Free Pass 3** | `tmp = 0x100; ptr = NULL; free(0x100);` | `0x300` | All heap blocks reclaimed; `ptr == NULL` exits `while` | Frees `0x100` (`1`) |

### 5.3.5 Linked List Operations Cheat Sheet

| Operation | Canonical C Idiom | Critical Safety Rule |
|---|---|---|
| **Define Node** | `typedef struct node { int number; struct node *next; } node;` | Inside the braces, `struct node *next;` is required because the `node` alias is not defined yet |
| **Empty List** | `node *list = NULL;` | Initialising the head pointer to `NULL` ensures end-of-list checks stop cleanly |
| **Prepend Node** | `n->next = list; list = n;` | `n->next` must be wired **before** `list = n` is reassigned, preventing orphaned nodes |
| **Traverse** | `for (node *ptr = list; ptr != NULL; ptr = ptr->next)` | Advancing uses `ptr = ptr->next` rather than `ptr++` because nodes are not contiguous in memory |
| **Free All** | `while (ptr != NULL) { node *tmp = ptr; ptr = ptr->next; free(tmp); }` | `ptr` must advance before `free(tmp)` executes so freed memory is never dereferenced |

### 5.3.6 Questions

1. In `list.c`, why must `node *ptr = list;` be declared again on line 35 right before the `while` loop, even though `node *ptr = list;` already appeared in the `for` loop on line 29?
2. What goes wrong if the `while` loop in `list.c` executes `free(tmp);` *before* `ptr = ptr->next;`?
3. When `./list` receives inputs `10`, `20` and `30`, in what exact order are the three numbers printed? In what exact order are the three nodes freed?
4. Why would replacing `ptr = ptr->next` with `ptr++` inside the traversal `for` loop fail on a linked list?

## 5.4 Hash Tables and Hash Function Principles

### 5.4.1 From a Single Linked List to an Array of Buckets

Storing `"Hey!"`, `"Hello!"` and `"Lo there!"` inside one flat linked list keeps memory overhead minimal because space is allocated only for entries actually added:

```text
+--------+     +----------+     +-------------+
| "Hey!" |---->| "Hello!" |---->| "Lo there!" |----> NULL
+--------+     +----------+     +-------------+
```

However, searching a single linked list runs in **`O(n)`** time.

In the worst case (when the target phrase sits at the very tail or is absent altogether), the program must visit and compare every single node in the list one by one.

A **hash table** accelerates search by replacing one long list with a fixed-size **array of linked lists**. Each array element is a **bucket** (`node *`) indexed by a non-negative integer (`0`, `1`, `2`, ...):

```text
Index  Bucket
+----+-------+
| .. |  ...  |
+----+-------+     +--------+     +----------+
|  7 |   H   |---->| "Hey!" |---->| "Hello!" |----> NULL
+----+-------+     +--------+     +----------+
|  8 |   I   |----> NULL
+----+-------+
|  9 |   J   |----> NULL
+----+-------+
| 10 |   K   |----> NULL
+----+-------+     +-------------+
| 11 |   L   |---->| "Lo there!" |----> NULL
+----+-------+     +-------------+
| .. |  ...  |
+----+-------+
```

Splitting entries across 26 alphabetical buckets exhibits the classic time-space trade-off:

- **Space cost:** Memory is reserved upfront for all 26 bucket pointers (`0` through `25`), even when buckets `I` (`8`), `J` (`9`) and `K` (`10`) remain completely empty.
- **Time benefit:** Searching for `"Hello!"` examines only bucket `7` (`H`) and skips the other 25 buckets entirely.

### 5.4.2 Example Hash Mappings: First Letter vs String Length

Because C arrays are indexed by integers (`0`, `1`, `2`, ...) rather than letters, a **hash function** sits at the front door of a hash table to convert an input key into a numeric bucket index.

#### 5.4.2.1 Mapping 1 — Alphabetical Index of the First Character

Under a first-letter rule (`'A'` → `0`, `'B'` → `1`, ..., `'H'` → `7`, ..., `'L'` → `11`, ..., `'Z'` → `25`), passing `"Hey!"` into the hash function outputs integer index `7`:

```text
             +-----------------+
  "Hey!" --->|  Hash Function  |---> 7
             +-----------------+
```

Both `"Hey!"` and `"Hello!"` begin with `'H'` and therefore map to bucket `7`. `"Lo there!"` begins with `'L'` and maps to bucket `11`.

#### 5.4.2.2 Mapping 2 — Character Length of the Input String

Switching the hash function changes how the exact same phrases distribute across buckets.

Consider a second hash function that maps each phrase to its character count:

```text
             +-----------------+
  "Hey!" --->|  Hash Function  |---> 4
             +-----------------+
```

Under a length-based mapping, the three phrases separate into distinct buckets instead of sharing bucket `H`:

```text
Index
+-----+
| ... |
+-----+     +--------+
|  4  |---->| "Hey!" |----> NULL        (4 characters: H-e-y-!)
+-----+     +--------+
|  5  |----> NULL
+-----+     +----------+
|  6  |---->| "Hello!" |----> NULL      (6 characters: H-e-l-l-o-!)
+-----+     +----------+
|  7  |----> NULL
+-----+     +-------------+
|  8  |---->| "Lo there!" |----> NULL   (8 non-space characters; 9 via raw strlen)
+-----+     +-------------+
| ... |
+-----+
```

### 5.4.3 Properties of a Good Hash Function and Collisions

When two distinct inputs produce the same hash value (such as `"Hey!"` and `"Hello!"` both hashing to `7`), a **collision** occurs. Both colliding items must then share the linked list at that bucket.

An ideal hash function satisfies three design criteria:

1. **Deterministic (same input always yields the same output):** Hashing `"Hey!"` must return the exact same bucket index every single time. Any randomness or drift would send a search lookup to a different bucket from where the word was originally inserted.
2. **Even distribution across buckets:** Inputs spread uniformly so every bucket's linked list stays as short as possible (ideally a single item or a very short list per bucket).
3. **Uses all buckets:** Every allocated slot in the bucket array can receive entries, avoiding wasted memory on unreachable or perpetually empty buckets.

Evaluating the two example hash functions against those three criteria reveals why neither qualifies as a *good* general-purpose hash function:

| Hash Function | Deterministic? | Even Distribution? | Uses All Buckets? | Verdict & Failure Mode |
|---|---|---|---|---|
| **First Letter (`0`–`25`)** | Yes ✓ | No — common initials (`H`, `S`, `T`) accumulate long chains | Weak — rare initials (`X`, `Z`) sit empty or nearly empty in small vocabularies | **Poor:** heavy clustering in common starting letters |
| **String Length** | Yes ✓ | No — most English words cluster tightly around 3–10 letters, causing massive collisions | No — buckets above the longest word's length (and `0`) sit permanently empty | **Poor:** severe collisions in common word lengths and wasted upper buckets |

### 5.4.4 Questions

1. What is the worst-case time complexity of searching a single linked list of `n` words? How does a hash table aim to improve on that runtime?
2. Why is determinism ("same input always gives the same output") a strict correctness requirement for a hash function rather than merely a performance goal?
3. What is a **collision** in a hash table? Why does hashing English words solely by word length lead to severe collisions?

## 5.5 Implementing and Improving a Hash Function (hash.c)

### 5.5.1 Single-Letter Hashing with ASCII Normalisation

The exercise program `hash.c` prompts the user for a word (`char *word`, recalling from Week 4 that CS50's `string` is an alias for `char *`). It then prints the integer bucket returned by `int hash(char *word)`.

Consider first a baseline `hash` function that maps a word's first letter (`word[0]`) case-insensitively onto bucket indices `0` through `25` (`'A'`/`'a'` → `0`, `'B'`/`'b'` → `1`, ..., `'Z'`/`'z'` → `25`).

In ASCII, uppercase and lowercase letters sit 32 positions apart:

- Uppercase `'A'` through `'Z'` occupy ASCII codes `65` through `90`.
- Lowercase `'a'` through `'z'` occupy ASCII codes `97` through `122`.

Rather than writing separate arithmetic branches for uppercase (`- 65`) and lowercase (`- 97`), calling `toupper()` from `<ctype.h>` normalises any lowercase letter to its uppercase ASCII equivalent first (leaving already-uppercase letters unchanged).

Subtracting `'A'` (`65`) then shifts `'A'`–`'Z'` (`65`–`90`) cleanly onto `0`–`25`:

```c
char c = toupper(word[0]);
return c - 'A';
```

Plugging concrete ASCII numbers into `c - 'A'` confirms the mapping:

- `"Apple"` → `word[0]` is `'A'` (`65`) → `toupper('A')` is `65` → `65 - 65 = 0`.
- `"cat"` → `word[0]` is `'c'` (`99`) → `toupper('c')` is `'C'` (`67`) → `67 - 65 = 2`.
- `"Cat"` → `word[0]` is `'C'` (`67`) → `toupper('C')` is `'C'` (`67`) → `67 - 65 = 2`.

### 5.5.2 Defensive Edge Cases, Prototypes and Whitespace

The two-line formula `toupper(word[0]) - 'A'` breaks or misbehaves on malformed inputs unless three defensive guards and compiler requirements are addressed.

#### 5.5.2.1 Edge Case 1 — Non-Alphabetical Characters (`isalpha`)

Consider an input starting with a digit or symbol, such as `"123"` (`'1'` has ASCII value `49`).

`toupper('1')` leaves `'1'` (`49`) unchanged. Evaluating `49 - 65` then returns `-16` — an index *before* the start of a bucket array that would crash or corrupt memory!

Checking `isalpha(c)` (from `<ctype.h>`) ensures the `c - 'A'` subtraction runs only on alphabetical letters.

For non-alphabetical characters, returning `0` would silently collide with `'A'` words. Returning a negative error code (`-1`) or routing to a dedicated 27th "other" bucket (`26`) cleanly separates non-letters from `'A'`.

> **Control-Flow Note (`if` Without `else`):** When an `if (isalpha(c))` block ends with `return c - 'A';`, writing `else` before the fallback `return -1;` is optional and does not alter control flow. Execution reaches the line after the `if` block *only* when the `if` condition was false.

#### 5.5.2.2 Edge Case 2 — `NULL` Pointer or Empty String (`""`)

Attempting to hash characters is invalid when `word` is `NULL` or when the user presses Enter without typing anything.

Pressing Enter alone produces an empty string `""` whose index `0` holds only the null terminator `'\0'` (`strlen(word) == 0`).

Guarding the top of `hash()` with `if (word == NULL || strlen(word) == 0)` (requiring `#include <string.h>` for `strlen`) catches missing input immediately. Returning `-2` distinguishes empty input (`-2`) from a non-alphabetical character (`-1`).

> **Short-Circuit Order Matters:** In `word == NULL || strlen(word) == 0`, the `word == NULL` check **must** appear on the left of `||`. Passing `NULL` into `strlen()` triggers a segmentation fault. C's left-to-right short-circuit evaluation skips `strlen(word)` entirely whenever `word == NULL` evaluates to true.

#### 5.5.2.3 Edge Case 3 — Leading Whitespace (`isspace` vs Python `.strip()`)

In R, calling `trimws(word)` removes leading and trailing spaces in a single function call. Python similarly provides the built-in `word.strip()` string method.

C strings (`char *`) have no built-in `.strip()` method.

A C hash function designed to ignore leading spaces must step past characters satisfying `isspace()` (from `<ctype.h>`). In `hash.c`, a leading space such as `" cat"` simply fails `isalpha(' ')` and returns `-1`.

#### 5.5.2.4 Compiler Prototypes and Complete Single-Letter `hash.c`

Defining `int hash(char *word)` *below* `main` without declaring its **prototype** `int hash(char *word);` above `main` causes compilation to fail with an `undeclared function 'hash'` error, because the C compiler reads source files top-to-bottom.

Putting the prototype, headers and guards together yields the complete single-letter version:

```c
#include <cs50.h>
#include <ctype.h>
#include <stdio.h>
#include <string.h>

int hash(char *word);

int main(void)
{
    char *word = get_string("Word: ");
    printf("Hash value: %i\n", hash(word));
}

int hash(char *word)
{
    if (word == NULL || strlen(word) == 0)
    {
        return -2;
    }

    char c = word[0];
    if (isalpha(c))
    {
        c = toupper(c);
        return c - 'A';
    }
    return -1;
}
```

Testing all four input cases verifies each branch:

- `"cat"` → `2`
- `"Cat"` → `2`
- `""` (pressing Enter with nothing typed) → `-2`
- `"123"` (non-alphabetical start) → `-1`

### 5.5.3 Extending hash.c to Two Letters and Avoiding Additive Collisions

Mapping solely by the first letter funnels every `'H'` word (`"Hey"`, `"Hello"`, `"Hi"`, `"House"`, ...) into one crowded bucket.

Examining the **first two letters** (`word[0]` and `word[1]`) subdivides each starting letter into finer buckets (`HA`, `HB`, ..., `HE`, ..., `HZ`).

#### 5.5.3.1 Guarding Two-Character Access

Reading `word[1]` requires at least two characters in `word` because a 1-letter word like `"a"` stores `'\0'` at `word[1]`. The length guard therefore tightens from `strlen(word) == 0` to `strlen(word) < 2`.

*(In a real spell-checker dictionary containing valid 1-letter words like `"a"` and `"I"`, 1-letter words must still map to a valid bucket rather than returning `-2`. Here `hash.c` returns `-2` to illustrate safe bounds checking before indexing `word[1]`.)*

#### 5.5.3.2 Why Simple Addition Fails (`"AC"` vs `"BB"`)

A tempting first attempt converts both letters to `0`–`25` and adds them directly:

```c
// FLAWED: different letter pairs collide!
return (c - 'A') + (c1 - 'A');
```

Addition alone is position-blind (commutative), so distinct two-letter prefixes collapse into the exact same sum:

- `"AC"` → `'A'` (`0`) + `'C'` (`2`) = **`2`**
- `"BB"` → `'B'` (`1`) + `'B'` (`1`) = **`2`**
- `"CA"` → `'C'` (`2`) + `'A'` (`0`) = **`2`**

Writing `(c + c1) - 'A'` suffers from the exact same flaw (`65 + 67 - 65 = 67` for `"AC"` versus `66 + 66 - 65 = 67` for `"BB"`).

#### 5.5.3.3 Positional Scaling (`* 100`) in Final `hash.c`

Multiplying the first letter's `0`–`25` value by `100` before adding the second letter's `0`–`25` value assigns each character its own non-overlapping decimal digits.

That scaling behaves like hotel room numbers: the first letter selects the floor (`000`, `100`, `200`, ..., `2500`) while the second letter selects the room (`00`–`25`):

```c
#include <cs50.h>
#include <ctype.h>
#include <stdio.h>
#include <string.h>

int hash(char *word);

int main(void)
{
    char *word = get_string("Word: ");
    printf("Hash value: %i\n", hash(word));
}

// Hash the word based on the first two letters of the word
int hash(char *word)
{
    // Word is less than 2 letters
    if (word == NULL || strlen(word) < 2)
    {
        return -2;
    }

    char c = word[0];
    char c1 = word[1];

    if (isalpha(c) && isalpha(c1))
    {
        c = toupper(c);
        c1 = toupper(c1);
        return (c - 'A') * 100 + c1 - 'A';
    }
    return -1;
}
```

Tracing `(c - 'A') * 100 + c1 - 'A'` across colliding pairs confirms that every two-letter combination now receives a unique integer:

| Input Word | `c` (`word[0]`) | `c1` (`word[1]`) | Calculation `(c - 'A') * 100 + (c1 - 'A')` | Returned Hash |
|---|---|---|---|---|
| `"a"` | — | — | `strlen("a") < 2` triggers guard | `-2` |
| `"1a"` | `'1'` | `'a'` | `isalpha('1')` is false | `-1` |
| `"aa"` | `'A'` (`0`) | `'A'` (`0`) | `0 * 100 + 0` | `0` |
| `"AC"` | `'A'` (`0`) | `'C'` (`2`) | `0 * 100 + 2` | `2` |
| `"BB"` | `'B'` (`1`) | `'B'` (`1`) | `1 * 100 + 1` | `101` |
| `"CA"` | `'C'` (`2`) | `'A'` (`0`) | `2 * 100 + 0` | `200` |
| `"cat"` | `'C'` (`2`) | `'A'` (`0`) | `2 * 100 + 0` | `200` |
| `"zz"` | `'Z'` (`25`) | `'Z'` (`25`) | `25 * 100 + 25` | `2525` |

> **Further Improvements for `speller`:** While `* 100` makes the two letter codes visually readable in base 10 (`101` clearly displays `1` and `01`), indices `26`–`99`, `126`–`199` and so forth sit permanently unused (multiplying by `26` instead of `100` packs all `26 * 26 = 676` pairs contiguously into `0`–`675`). Stronger hash functions for `speller` go further by combining ASCII values across **all** characters of the word (weighted by position and bounded with `% N`) so words sharing a two-letter prefix like `"cat"`, `"cab"` and `"camera"` also spread across separate buckets.

### 5.5.4 Hash Function Design Cheat Sheet

| Check / Step | C Idiom | Purpose |
|---|---|---|
| **Null & Length Guard** | `if (word == NULL \|\| strlen(word) < 2)` | Prevents `NULL` dereference (via `\|\|` short-circuiting) and out-of-bounds reads |
| **Alphabetical Check** | `if (isalpha(c) && isalpha(c1))` | Prevents digits, spaces or punctuation from producing negative bucket indices |
| **Case Normalisation** | `c = toupper(c);` | Collapses `'a'`–`'z'` (`97`–`122`) onto `'A'`–`'Z'` (`65`–`90`) so `"Cat"` and `"cat"` match |
| **Zero-Based Letter Code** | `c - 'A'` | Shifts ASCII `65`–`90` onto `0`–`25` |
| **Positional Weighting** | `(c - 'A') * 100 + (c1 - 'A')` (or `* 26`) | Separates `"AC"` from `"BB"` so transposed or equal-sum letter pairs do not collide |

### 5.5.5 Questions

1. Why does `hash.c` call `toupper(c)` before computing `c - 'A'`? What integer would `'a' - 'A'` produce without `toupper`?
2. What happens if `if (strlen(word) < 2 || word == NULL)` is written in reverse order when `word` is `NULL`?
3. Why does `(c - 'A') + (c1 - 'A')` fail to give every two-letter prefix a unique bucket? How does multiplying `(c - 'A')` by `100` (or `26`) eliminate that collision?
4. Under the final `hash.c` implementation, what exact values are returned by `hash("Hi")`, `hash("I")` and `hash("hi!")`?

## 5.6 Answers

### 5.1 Answers

**1.** **Search** ranks first, **insertion** second and **deletion** third.

Spoken audio is checked continuously against a static 500-phrase list, so search latency directly governs assistant responsiveness, whereas insertions and deletions occur only during rare software updates.

A plain linked list requires `O(n)` linear traversal from the head node on every lookup, making it poorly suited to a search-dominated workload compared with a hash table or trie.

**2.** Two nodes below the root store `"HE"`: one node for `'H'` leading to one node for `'E'`.

In a trie, words sharing a common prefix share the exact same upper path (`root -> H -> E`) before branching into `'L' -> 'L' -> 'O'` (for `"HELLO"`) and `'Y'` (for `"HEY"`).

**3.** Increasing bucket count in a hash table or switching to a trie trades **space (memory)** for **time (speed)**.

Allocating more buckets or wide character-pointer arrays consumes substantially more heap memory (often megabytes, including many unused `NULL` pointers) in exchange for shorter or zero-collision search paths.

### 5.2 Answers

**1.** C reads declarations from top to bottom.

Inside the curly braces of `typedef struct node { ... } node;`, the shorthand alias `node` does not exist yet — it is created only at the closing `} node;` semicolon, so `struct node *next;` must use the structure tag declared on line 1.

**2.** An uninitialised local pointer `node *list;` holds an arbitrary **garbage address** left over on the stack.

Without `NULL`, there is no way to recognise that the list is empty. On the first insertion, `n->next = list;` would copy that garbage address into the tail node instead of marking a clean `NULL` end boundary.

**3.** On iteration `i = 1`, `list` initially holds the address of node 1 (say `0x100`) while `n` holds the address of node 2 (say `0x200`).

Executing `list = n;` first overwrites `list` with `0x200`, destroying the only pointer to `0x100` (orphaning node 1 as a memory leak).

The next line `n->next = list;` then points `0x200`'s `next` field back at `0x200` itself, creating an infinite self-loop.

**4.** Because `list` is initialised to `NULL` before the loop begins, on iteration `i = 0` the statement `n->next = list;` evaluates to `n->next = NULL;`.

That single line marks the first allocated node as the end of the chain using the exact same logic that links subsequent nodes ahead of an existing head.

### 5.3 Answers

**1.** Variable **scope** restricts the first `ptr` (`for (node *ptr = list; ...)` on line 29) to that `for` loop alone. Once the loop finishes, that `ptr` ceases to exist.

Declaring `node *ptr = list;` on line 35 creates a fresh cursor variable in `main`'s scope, reset to the head of the list so the `while` loop can walk and free every node.

**2.** Inside the `while` loop, `tmp` and `ptr` initially point to the same current node (`node *tmp = ptr;`).

Calling `free(tmp);` *before* `ptr = ptr->next;` deallocates that heap block first, causing `ptr = ptr->next;` to read `next` out of already-freed memory — a **use-after-free** bug.

**3.** Head insertion prepends each newcomer at the front, so entering `10`, `20` and `30` constructs the chain `30 -> 20 -> 10 -> NULL`.

Both the `for` print loop and the `while` free loop begin at `list` (the head), so the numbers print in LIFO order (`30`, `20`, `10`) and the nodes are freed in that exact same order (`30`, `20`, `10`).

**4.** Pointer arithmetic `ptr++` steps forward by `sizeof(node)` bytes to the immediately adjacent memory address, which is valid only for contiguous arrays.

Linked-list nodes come from separate `malloc` calls and may sit anywhere on the heap. Only `ptr->next` stores the actual address of the next node.

### 5.4 Answers

**1.** Searching a single linked list of `n` words takes **`O(n)`** time in the worst case because every node from head to tail may need to be inspected sequentially.

A hash table divides those `n` words across an array of buckets via a hash function so a search jumps straight to one bucket index and scans only that bucket's much shorter linked list.

**2.** During insertion, a word's hash value chooses which bucket stores it. During search, the hash table hashes the word again and inspects *only* the returned bucket index.

Were the hash function to return a different index on the second call, the search would look in the wrong bucket and falsely report a stored word as missing.

**3.** A **collision** occurs when two or more distinct inputs produce the same bucket index from the hash function, forcing them to share that bucket's linked list.

Hashing by string length causes severe collisions because natural-language words cluster tightly around common lengths (such as 4 to 8 characters), piling thousands of words into a handful of buckets while leaving longer-length buckets empty.

### 5.5 Answers

**1.** Uppercase and lowercase ASCII letters have different numeric codes: `'A'` is `65`, whereas `'a'` is `97`.

Without `toupper(c)`, `'a' - 'A'` evaluates to `97 - 65 = 32` (outside `0`–`25` and separate from `'A'`). Calling `toupper(c)` converts `'a'`–`'z'` to `'A'`–`'Z'` (`65`–`90`) first so both cases map identically onto `0`–`25`.

**2.** C evaluates `||` from left to right.

Placing `strlen(word) < 2` on the left when `word` is `NULL` calls `strlen(NULL)`, dereferencing address `0x0` and crashing with a **segmentation fault** before `word == NULL` is ever checked.

**3.** Simple addition `(c - 'A') + (c1 - 'A')` gives both character positions equal weight, so `"AC"` (`0 + 2 = 2`), `"BB"` (`1 + 1 = 2`) and `"CA"` (`2 + 0 = 2`) all collide in bucket `2`.

Multiplying `(c - 'A')` by `100` (or `26`) assigns a distinct positional weight to the first letter (`"AC"` → `2`, `"BB"` → `101`, `"CA"` → `200`), giving every two-letter pair a unique index.

**4.** Evaluating each call against `hash.c`:

- `hash("Hi")`: `strlen("Hi")` is `2` (`>= 2`) and both `'H'` and `'i'` pass `isalpha`. Normalising with `toupper` yields `'H'` (`7`) and `'I'` (`8`), so `(7 * 100) + 8` returns **`708`**.
- `hash("I")`: `strlen("I")` is `1` (`< 2`), triggering the length guard and returning **`-2`**.
- `hash("hi!")`: `strlen("hi!")` is `3` (`>= 2`). Because `hash.c` inspects only `word[0]` (`'h'`) and `word[1]` (`'i'`) while ignoring `word[2]` (`'!'`), `(7 * 100) + 8` again returns **`708`**.